# Moroccan Players WC26 Scraping

Multi-source football data enrichment pipeline for the Morocco World Cup 2026 squad.

Notebook style follows the Edd Webster scraping notebooks: dependencies, project brief, scraping sections, unification, export and summary.

## 1. Notebook Dependencies

In [ ]:
import platform
import sys
assert sys.version_info >= (3, 10)

import pandas as pd
import numpy as np
from pathlib import Path

from moroccan_players_scraper import (
    FBREF_OUT, TRANSFERMARKT_OUT, CAPOLOGY_OUT, WHOSCORED_OUT, MASTER_OUT,
    scrape_fbref, scrape_transfermarkt, scrape_capology, scrape_whoscored, merge_all_sources
)

print(f'Python: {platform.python_version()}')
print(f'pandas: {pd.__version__}')

## 2. Project Brief ? Morocco WC2026 Squad ? Multi-source data enrichment pipeline

Goal: enrich the 29-player Morocco squad seed with public football data from FBref, TransferMarkt, Capology and WhoScored. Missing data remains `N/A`; the pipeline never fabricates values.

## 3. Data Scraping

### 3.1 FBref Player Stats

Scrapes player-level tables where configured. Handles unavailable advanced metrics after the 2026 Stats Perform/FBref access changes by logging warnings and preserving basic available fields.

In [ ]:
df_fbref = scrape_fbref()
df_fbref.head()

### 3.2 TransferMarkt Bio & Market Value

In [ ]:
df_tm = scrape_transfermarkt()
df_tm.head()

### 3.3 Capology Salary Estimates

In [ ]:
df_capology = scrape_capology()
df_capology.head()

### 3.4 WhoScored Event Summary

In [ ]:
df_whoscored = scrape_whoscored()
df_whoscored.head()

## 4. Data Unification ? merge + confidence scoring

In [ ]:
df_master = merge_all_sources()
df_master[['player_id', 'player_name', 'data_confidence']].head()

## 5. Data Export

In [ ]:
outputs = {
    'FBref': FBREF_OUT,
    'TransferMarkt': TRANSFERMARKT_OUT,
    'Capology': CAPOLOGY_OUT,
    'WhoScored': WHOSCORED_OUT,
    'Master': MASTER_OUT,
}
for name, path in outputs.items():
    print(f'{name}: {path} exists={Path(path).exists()}')

## 6. Summary

In [ ]:
summary_rows = []
for source, path, key_fields in [
    ('FBref', FBREF_OUT, 'Standard, shooting, passing, defense, possession'),
    ('TransferMarkt', TRANSFERMARKT_OUT, 'bio, club, contract, market value'),
    ('Capology', CAPOLOGY_OUT, 'salary, contract end'),
    ('WhoScored', WHOSCORED_OUT, 'rating, goals, assists, defensive events'),
]:
    if Path(path).exists():
        df = pd.read_csv(path)
        found = int((~df.astype(str).isin(['N/A', '', 'nan'])).any(axis=1).sum())
    else:
        found = 0
    summary_rows.append({'Source': source, 'Players found': found, 'Key fields': key_fields, 'Output file': str(path)})

pd.DataFrame(summary_rows)